# Train Model Notebook
This notebook illustrates a two-step modeling process: first a simple baseline model, then a more flexible Random Forest. The goal is to teach students how to compare model performance and choose improvements.


## 1. Load processed data
Load the cleaned dataset produced by the prepare notebook and inspect the available features.


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from seattle_energy.model_training import load_dataset, get_feature_columns, TARGET_COLUMN
from sklearn.model_selection import train_test_split

df = load_dataset()
features = get_feature_columns(df)
print(f"Loaded dataset with {len(df)} rows and {len(features)} feature columns.")
print("Target column:", TARGET_COLUMN)
df.head()


## 2. Baseline model: linear regression on log surface
A baseline model helps students understand whether the advanced model is actually better. We use a simple linear regression on the logarithm of building surface area.


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

baseline_feature = ["log_surface"]
if not set(baseline_feature).issubset(df.columns):
    raise ValueError("Required baseline feature not found in dataset")

X = df[baseline_feature]
y = df[TARGET_COLUMN]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

baseline_model = LinearRegression()
baseline_model.fit(X_train, y_train)
y_pred_train = baseline_model.predict(X_train)
y_pred_test = baseline_model.predict(X_test)

def evaluate(y_true, y_pred, name=""):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    print(f"{name}: R2={r2:.4f}, MAE={mae:.2f}, RMSE={rmse:.2f}")

evaluate(y_train, y_pred_train, "Baseline Train")
evaluate(y_test, y_pred_test, "Baseline Test")


## 3. Random Forest with the full feature set
Now use the full set of engineered features and a more flexible model. This section follows the existing `model_training.py` logic while making the training process visible in the notebook.


In [ ]:
from seattle_energy.model_training import build_search_model, save_bento_model

full_features = [col for col in features if col not in ["OSEBuildingID", TARGET_COLUMN]]
X = df[full_features]
y = df[TARGET_COLUMN]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

search = build_search_model()
search.fit(X_train, y_train)
best_model = search.best_estimator_
print("Best hyperparameters:", search.best_params_)

evaluate(y_train, best_model.predict(X_train), "Random Forest Train")
evaluate(y_test, best_model.predict(X_test), "Random Forest Test")


## 4. Save the model for serving
The final step exports the best trained model to the BentoML store, ready for deployment.


In [ ]:
save_bento_model(best_model, full_features)
print("Saved BentoML model as random_forest_energy:latest")
